B1: Cấu hình & Tải Dữ liệu + FastText

In [1]:
import pandas as pd
import numpy as np
import re
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Embedding, Conv1D, GlobalMaxPooling1D, Dense, Dropout, Concatenate
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.utils.class_weight import compute_class_weight

# [THÔNG BÁO HỆ THỐNG]
print("[INFO] Đang tải bộ dữ liệu FastText (Wiki News 300d)...")
# Tải FastText (Nặng ~600MB)
!wget -N https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip
print("[INFO] Đang giải nén dữ liệu FastText...")
!unzip -n -q wiki-news-300d-1M.vec.zip

# Load dataset MBTI
print("[INFO] Đang nạp dữ liệu từ tập tin 'mbti_1.csv'...")
try:
    df = pd.read_csv('mbti_1.csv', on_bad_lines='skip', engine='python')
    print(f"[SUCCESS] Tải dữ liệu MBTI hoàn tất. Tổng số bản ghi: {len(df)}")
except FileNotFoundError:
    print("[ERROR] Không tìm thấy file 'mbti_1.csv'. Vui lòng tải file lên Colab trước khi chạy.")

[INFO] Đang tải bộ dữ liệu FastText (Wiki News 300d)...
--2025-12-12 16:02:31--  https://dl.fbaipublicfiles.com/fasttext/vectors-english/wiki-news-300d-1M.vec.zip
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 18.154.144.13, 18.154.144.102, 18.154.144.87, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|18.154.144.13|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 681808098 (650M) [application/zip]
Saving to: ‘wiki-news-300d-1M.vec.zip’

wiki-news-300d-1M.v 100%[===================>] 650.22M   287MB/s    in 2.3s    

2025-12-12 16:02:34 (287 MB/s) - ‘wiki-news-300d-1M.vec.zip’ saved [681808098/681808098]

[INFO] Đang giải nén dữ liệu FastText...
[INFO] Đang nạp dữ liệu từ tập tin 'mbti_1.csv'...
[SUCCESS] Tải dữ liệu MBTI hoàn tất. Tổng số bản ghi: 8675


B2: Tiền xử lý & Tokenization

In [2]:
# Cấu hình siêu tham số
MAX_NB_WORDS = 20000       # Số lượng từ vựng tối đa
MAX_SEQUENCE_LENGTH = 500  # Độ dài cố định của câu
EMBEDDING_DIM = 300        # FastText dùng vector 300 chiều

# 1. Hàm làm sạch dữ liệu
def clean_text_formal(text):
    """
    Hàm chuẩn hóa văn bản:
    - Chuyển về chữ thường.
    - Loại bỏ URL.
    - Loại bỏ các từ khóa MBTI (Data Leakage Prevention).
    """
    text = str(text).lower()
    text = re.sub(r'http\S+', '', text)

    # Danh sách từ khóa MBTI cần loại bỏ
    mbti_types = ['infj', 'entp', 'intp', 'intj', 'entj', 'enfj', 'infp', 'enfp',
                  'isfp', 'istp', 'isfj', 'istj', 'estp', 'esfp', 'estj', 'esfj']
    pattern = r'\b(?:' + '|'.join(mbti_types) + r')\b'
    text = re.sub(pattern, '', text)

    text = re.sub(r'[^a-z\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("[INFO] Đang thực hiện làm sạch dữ liệu văn bản...")
df['clean_posts'] = df['posts'].apply(clean_text_formal)

# 2. Mã hóa nhãn (Binary Encoding)
def get_binary_labels(mbti_type):
    return pd.Series({
        'IE': 1 if 'I' in mbti_type else 0,
        'NS': 1 if 'N' in mbti_type else 0,
        'TF': 1 if 'T' in mbti_type else 0,
        'JP': 1 if 'J' in mbti_type else 0
    })
binary_targets = df['type'].apply(get_binary_labels)
df = pd.concat([df, binary_targets], axis=1)

# 3. Phân chia tập dữ liệu & Tokenize
print("[INFO] Đang phân chia tập Train/Test và mã hóa văn bản...")
X_train_text, X_test_text, y_train_multi, y_test_multi = train_test_split(
    df['clean_posts'], df[['IE', 'NS', 'TF', 'JP']], test_size=0.2, random_state=42, stratify=df['type']
)

tokenizer = Tokenizer(num_words=MAX_NB_WORDS, oov_token='<OOV>')
tokenizer.fit_on_texts(X_train_text)

X_train_seq = tokenizer.texts_to_sequences(X_train_text)
X_test_seq = tokenizer.texts_to_sequences(X_test_text)

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_SEQUENCE_LENGTH)
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_SEQUENCE_LENGTH)

print(f"[SUCCESS] Tiền xử lý hoàn tất. Kích thước tập huấn luyện: {X_train_pad.shape}")

[INFO] Đang thực hiện làm sạch dữ liệu văn bản...
[INFO] Đang phân chia tập Train/Test và mã hóa văn bản...
[SUCCESS] Tiền xử lý hoàn tất. Kích thước tập huấn luyện: (6940, 500)


B3: Load FastText & Tạo Ma Trận Embedding

In [3]:
def load_fasttext_embeddings(word_index, max_words, embedding_dim=300):
    """
    Hàm đọc file vector FastText và ánh xạ vào từ điển Tokenizer.
    """
    print("[INFO] Đang nạp vector từ file 'wiki-news-300d-1M.vec'...")
    embeddings_index = {}

    # Đọc file FastText (Bỏ dòng đầu tiên chứa meta-data)
    with open('wiki-news-300d-1M.vec', encoding='utf-8', errors='ignore') as f:
        for i, line in enumerate(f):
            if i == 0: continue # Bỏ qua header
            values = line.split()
            word = values[0]
            try:
                coefs = np.asarray(values[1:], dtype='float32')
                embeddings_index[word] = coefs
            except ValueError:
                continue

    print(f"[INFO] Đã nạp {len(embeddings_index)} vector từ vựng vào bộ nhớ.")

    # Xây dựng ma trận
    print("[INFO] Đang khởi tạo Ma trận Embedding...")
    embedding_matrix = np.zeros((max_words, embedding_dim))
    hits = 0
    misses = 0

    for word, i in word_index.items():
        if i < max_words:
            embedding_vector = embeddings_index.get(word)
            if embedding_vector is not None:
                embedding_matrix[i] = embedding_vector
                hits += 1
            else:
                misses += 1

    print(f"[THỐNG KÊ] Số từ khớp FastText: {hits}")
    print(f"[THỐNG KÊ] Số từ mới (cần học thêm): {misses}")
    return embedding_matrix

# Gọi hàm để tạo ma trận
fasttext_embedding_matrix = load_fasttext_embeddings(tokenizer.word_index, MAX_NB_WORDS, EMBEDDING_DIM)

[INFO] Đang nạp vector từ file 'wiki-news-300d-1M.vec'...
[INFO] Đã nạp 999994 vector từ vựng vào bộ nhớ.
[INFO] Đang khởi tạo Ma trận Embedding...
[THỐNG KÊ] Số từ khớp FastText: 19287
[THỐNG KÊ] Số từ mới (cần học thêm): 712


B4: Huấn luyện TextCNN + Class Weights

In [4]:
# Hàm dựng TextCNN
def build_fasttext_cnn_model():
    inputs = Input(shape=(MAX_SEQUENCE_LENGTH,))

    # Embedding Layer với FastText
    embedding = Embedding(MAX_NB_WORDS, EMBEDDING_DIM,
                          weights=[fasttext_embedding_matrix],
                          trainable=True)(inputs)

    # 3 Nhánh tích chập song song (Kernel size 3, 4, 5)
    conv_3 = Conv1D(filters=128, kernel_size=3, activation='relu')(embedding)
    pool_3 = GlobalMaxPooling1D()(conv_3)

    conv_4 = Conv1D(filters=128, kernel_size=4, activation='relu')(embedding)
    pool_4 = GlobalMaxPooling1D()(conv_4)

    conv_5 = Conv1D(filters=128, kernel_size=5, activation='relu')(embedding)
    pool_5 = GlobalMaxPooling1D()(conv_5)

    # Gộp đặc trưng
    merged = Concatenate()([pool_3, pool_4, pool_5])

    # Lớp phân loại
    dense = Dense(128, activation='relu')(merged)
    drop = Dropout(0.3)(dense)
    outputs = Dense(1, activation='sigmoid')(drop)

    model = Model(inputs=inputs, outputs=outputs)
    model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

# Hàm tính Class Weights
def calculate_class_weights(y_train_data):
    weights = compute_class_weight('balanced', classes=np.unique(y_train_data), y=y_train_data)
    return dict(enumerate(weights))

target_cols = ['IE', 'NS', 'TF', 'JP']
EPOCHS = 20
BATCH_SIZE = 64

print("="*60)
print("[INFO] BẮT ĐẦU HUẤN LUYỆN: TextCNN + FASTTEXT + CLASS WEIGHTS")
print("="*60)

for col in target_cols:
    print(f"\n[INFO] Đang huấn luyện trục: {col}...")

    y_train = y_train_multi[col].values
    y_test = y_test_multi[col].values

    # Tính Class Weights
    cls_weights = calculate_class_weights(y_train)

    # Build & Train
    model = build_fasttext_cnn_model()
    early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    model.fit(X_train_pad, y_train,
              epochs=EPOCHS, batch_size=BATCH_SIZE,
              validation_split=0.1,
              callbacks=[early_stop],
              class_weight=cls_weights,
              verbose=1)

    # Đánh giá
    y_pred = (model.predict(X_test_pad) > 0.5).astype(int)
    print(f"[KẾT QUẢ] Độ chính xác (Accuracy) cho trục {col}: {accuracy_score(y_test, y_pred):.2%}")
    print(classification_report(y_test, y_pred, digits=4))

print("\n[SUCCESS] QUÁ TRÌNH THỰC NGHIỆM ĐÃ HOÀN TẤT.")

[INFO] BẮT ĐẦU HUẤN LUYỆN: TextCNN + FASTTEXT + CLASS WEIGHTS

[INFO] Đang huấn luyện trục: IE...
Epoch 1/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 15s 85ms/step - accuracy: 0.4994 - loss: 0.7155 - val_accuracy: 0.7334 - val_loss: 0.6696
Epoch 2/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 3s 33ms/step - accuracy: 0.6202 - loss: 0.6659 - val_accuracy: 0.7507 - val_loss: 0.5875
Epoch 3/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - accuracy: 0.8374 - loss: 0.4552 - val_accuracy: 0.6801 - val_loss: 0.5952
Epoch 4/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - accuracy: 0.9776 - loss: 0.1179 - val_accuracy: 0.7651 - val_loss: 0.6514
Epoch 5/20
98/98 ━━━━━━━━━━━━━━━━━━━━ 3s 32ms/step - accuracy: 0.9996 - loss: 0.0152 - val_accuracy: 0.7622 - val_loss: 0.7306
55/55 ━━━━━━━━━━━━━━━━━━━━ 2s 16ms/step
[KẾT QUẢ] Độ chính xác (Accuracy) cho trục IE: 76.71%
              precision    recall  f1-score   support

           0     0.4941    0.3117    0.3823       401
           1     0.8138    0.9040    0.8565      1334

    accu